# 성적 확인용 해시(Hash) 조회기

성적을 "이름 — 점수" 형태로 공개하면 내 점수를 확인하려는 사람도, 다른 사람 점수를
구경하려는 사람도 똑같이 전체 명단을 보게 됩니다. 이 노트북은 그 대신 **이름 + 학번을
넣으면 항상 똑같이 나오는 "해시값"**을 계산해줍니다 — 성적 게시판에는 이름 대신
이 해시값을 "해시값 — 점수" 형태로 공개하면, 각자 자기 이름과 학번으로 자기 줄만
찾을 수 있고, 지나가다 명단을 봐도 누가 몇 점인지는 바로 알 수 없습니다.

## Hash function이란?

**해시 함수**는 어떤 입력(여기서는 "학번 + 이름")을 받아 고정된 길이의 문자열로
바꿔주는 함수입니다. 이번 실습에 쓰는 SHA-256은 다음 성질을 가집니다:

- **결정적(deterministic)**: 같은 입력을 넣으면 **언제, 몇 번을 다시 계산해도** 항상
  같은 해시값이 나옵니다.
- **되돌릴 수 없음(one-way)**: 해시값만 보고 원래 입력(이름, 학번)을 역으로 알아낼
  수 없습니다.
- **눈사태 효과(avalanche effect)**: 입력이 단 한 글자만 달라져도 해시값은 완전히
  다르게 나옵니다 — 그래서 오타가 있으면 전혀 다른(엉뚱한) 해시가 나옵니다.

In [ ]:
import hashlib

def compute_score_hash(name: str, student_id: str) -> str:
    name = name.strip()
    student_id = student_id.strip()
    if not name or not student_id:
        raise ValueError("이름과 학번을 모두 입력해주세요.")
    combined = f"{student_id}|{name}"
    return hashlib.sha256(combined.encode("utf-8")).hexdigest()


# 결정적(deterministic)이라는 걸 확인: 같은 입력을 두 번 계산해도 같은 값이 나와야 합니다.
h1 = compute_score_hash("홍길동", "20231234")
h2 = compute_score_hash("홍길동", "20231234")
assert h1 == h2
print("같은 입력 -> 항상 같은 해시:")
print(h1)

## 눈사태 효과 확인해보기

학번 끝자리 하나만 다른 두 입력의 해시를 나란히 비교해봅니다 — 입력은 거의 똑같은데
해시값은 전혀 다르게 나오는 걸 볼 수 있습니다. 즉, 학번이나 이름을 **정확히** 입력해야
게시판의 올바른 줄을 찾을 수 있습니다.

In [ ]:
print("20231234:", compute_score_hash("홍길동", "20231234"))
print("20231235:", compute_score_hash("홍길동", "20231235"))

## 직접 해보기

아래에 **본인의 이름과 학번**을 입력하고 "해시 계산하기" 버튼을 누르세요. 전체
해시값과, 게시판에 실제로 올라갈 **짧은 해시값**(앞 10자리)을 함께 보여줍니다.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

SHORT_HASH_LENGTH = 10

name_input = widgets.Text(description="이름:", placeholder="예: 홍길동")
id_input = widgets.Text(description="학번:", placeholder="예: 20231234")
compute_button = widgets.Button(description="해시 계산하기", button_style="primary")
output_area = widgets.Output()


def on_compute_clicked(_):
    with output_area:
        clear_output()
        try:
            full_hash = compute_score_hash(name_input.value, id_input.value)
        except ValueError as e:
            print(f"오류: {e}")
            return
        print(f"이름: {name_input.value.strip()}")
        print(f"학번: {id_input.value.strip()}")
        print(f"\n전체 해시값 (SHA-256):\n{full_hash}")
        print(f"\n짧은 해시값 (게시판에는 보통 이걸 씁니다, 앞 {SHORT_HASH_LENGTH}자리):\n{full_hash[:SHORT_HASH_LENGTH]}")


compute_button.on_click(on_compute_clicked)
display(name_input, id_input, compute_button, output_area)

## 주의할 점

- 이 방식이 막아주는 건 **"그냥 지나가다 보게 되는" 노출**입니다 — 명단에 이름이
  아니라 해시값만 있으니, 옆 사람 점수가 몇 점인지 한눈에 알아볼 수 없습니다.
- 하지만 누군가 **내 이름과 학번을 정확히 알고** 이 노트북에 그대로 입력하면, 저와
  **똑같은 해시값**이 나옵니다 — 해시는 입력이 같으면 항상 같은 값을 내놓기 때문입니다.
  즉, 이름과 학번을 아는 사람으로부터의 보호는 아니고, "모르는 사람들 사이에서 점수가
  우연히 노출되는 것"을 막는 용도입니다.

## (선택) 강사용: 전체 명단 한 번에 변환하기

강사가 "학번, 이름, 점수"로 된 전체 명단을 가지고 있다면, 위와 똑같은
`compute_score_hash` 함수로 "짧은 해시, 점수"만 담은 공개용 표를 만들 수 있습니다
(이름 열은 빼고 저장하므로, 공개해도 누가 몇 점인지 알 수 없습니다). 아래는 가상의
학생 5명으로 만든 예시입니다 — 실제로 쓸 때는 `roster` 부분을 진짜 명단으로
바꾸면 됩니다.

In [ ]:
import pandas as pd

# 예시용 가상 명단 (실제로 쓸 때는 이 부분을 진짜 학번/이름/점수로 바꾸세요)
roster = pd.DataFrame({
    "학번": ["20231001", "20231002", "20231003", "20231004", "20231005"],
    "이름": ["김민준", "이서연", "박도윤", "최지우", "정하은"],
    "점수": [88, 92, 76, 95, 81],
})

roster["해시"] = roster.apply(
    lambda row: compute_score_hash(row["이름"], row["학번"])[:SHORT_HASH_LENGTH], axis=1
)

print("전체 명단 (강사만 보관):")
print(roster)

public_table = roster[["해시", "점수"]].sort_values("해시").reset_index(drop=True)
print("\n공개용 표 (게시판에 올리는 것 -- 이름/학번 없이 해시와 점수만):")
print(public_table)